# MLE — Train Arcade RL Agent on Colab GPU

Two modes:
- **Retro** (fast, ~1000 FPS): Genesis/console ports via stable-retro
- **MAME** (slower, ~30 FPS): Original arcade ROMs via MAME pipes

**Runtime**: Select `2025.07` (Python 3.11) under Runtime > Change runtime type.

In [ ]:
# 1. Setup
!apt-get update -qq && apt-get install -y -qq mame xvfb > /dev/null 2>&1
!pip install -q sheeprl gymnasium pillow wandb opencv-python-headless 'numpy<2' stable-retro

import subprocess, os
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1x1x24'])
os.environ['DISPLAY'] = ':99'
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['XDG_RUNTIME_DIR'] = '/tmp'
if '/usr/games' not in os.environ.get('PATH', ''):
    os.environ['PATH'] = '/usr/games:' + os.environ['PATH']

!rm -rf /content/mle && git clone https://github.com/lilwg/mle.git /content/mle
%cd /content/mle

import torch, retro
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'stable-retro games: {len(retro.data.list_games())}')
print('Setup complete!')

## Option A: Retro (fast)
Uses stable-retro with Genesis/console ROMs. No MAME needed.
Upload the Genesis ROM (e.g. Frogger Genesis `.md` file).

In [ ]:
# 2a. Import ROM for stable-retro
from google.colab import files
uploaded = files.upload()

# Import into stable-retro
import os, tempfile
tmpdir = tempfile.mkdtemp()
for fname in uploaded:
    os.rename(fname, os.path.join(tmpdir, fname))
!python -m stable_retro.import {tmpdir}

# List available games with ROMs
import retro
games_with_roms = [g for g in retro.data.list_games() if retro.data.get_romfile_path(g)]
print(f'Games with ROMs: {games_with_roms}')

In [ ]:
# 3a. Train Dreamer v3 with stable-retro (FAST)
GAME = 'Frogger-Genesis'  # Change to your game
TIMESTEPS = 500_000

!cd /content/mle && python -u retro_train.py {GAME} --timesteps {TIMESTEPS}

## Option B: MAME (original arcade ROMs)
Uses MAME emulator via pipes. Slower but supports any arcade game.
Upload the MAME ROM zip (e.g. `frogger.zip`).

In [ ]:
# 2b. Upload MAME ROM
from google.colab import files
import os

uploaded = files.upload()
os.makedirs('roms', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'roms/{fname}')
rom_name = [f[:-4] for f in os.listdir('roms') if f.endswith('.zip')][0]
print(f'Game: {rom_name}')
!mame -rompath roms {rom_name} -verifyroms 2>&1

In [ ]:
# 3b. Train Dreamer v3 with MAME
GAME = rom_name
TIMESTEPS = 500_000

!cd /content/mle && \
    DISPLAY=:99 SDL_VIDEODRIVER=dummy SDL_AUDIODRIVER=dummy \
    MLE_ROMS_PATH=/content/mle/roms \
    python -u dreamer_train.py {GAME} --timesteps {TIMESTEPS}